Pricing Data Processing

In [0]:
from pyspark.sql.functions import *

In [0]:
%run /Workspace/consolidated_pipeline/02_dimension_data_processing/utilities

In [0]:
dbutils.widgets.text('catalog','fmcg','Catalog')
dbutils.widgets.text('data_source','gross_price','Data Source')

catalog=dbutils.widgets.get('catalog')
data_source=dbutils.widgets.get('data_source')

In [0]:
base_path=f"/Volumes/{catalog}/{bronze_schema}/sports_volume/{data_source}.csv"

In [0]:
df=spark.read.format('csv').option('header',True).option('inferSchema',True).load(base_path).\
    withColumn('read_time_stamp',current_timestamp())

In [0]:
display(df)

In [0]:
df.write.format('delta').option('mergeSchema','true').option('delta.enableChangeDataFeed','true').\
    mode('overwrite').saveAsTable(f'{catalog}.{bronze_schema}.{data_source}')

Getting Data in Bronze Layer

In [0]:
df_silver=spark.sql(f"select * from {catalog}.{bronze_schema}.{data_source}")
display(df_silver.limit(10))

change date format(month)

In [0]:
df_silver=df_silver.withColumn('month',
                     coalesce(
                        try_to_date(col('month'),'yyyy/MM/dd'),
                        try_to_date(col('month'),'dd/MM/yyyy'),
                        try_to_date(col('month'),'yyyy-MM-dd'),
                        try_to_date(col('month'),'dd-MM-yyyy'),
                     )
                     )
display(df_silver)

handle the gross price

In [0]:

df_silver = df_silver.withColumn(
    "gross_price",
    when(
        col("gross_price").rlike(r"^-?\d+(\.\d+)?$"),
        abs(col("gross_price").cast("double"))
    ).otherwise(0)
)

df_silver.show()

In [0]:
df_products=spark.table(f'{catalog}.{silver_schema}.products')

In [0]:
display(df_products)

In [0]:
df_joined=df_silver.join(df_products.select('product_code','product_id'),on='product_id',how='inner')
display(df_joined)

In [0]:
df_joined=df_joined.select('product_id','product_code','month','gross_price','read_time_stamp')
display(df_joined)

In [0]:
df_joined.write.format('delta').option('mergeSchema','true').option('delta.enableChangeDataFeed','true').\
    mode('overwrite').saveAsTable(f"{catalog}.{silver_schema}.{data_source}")

Gold

In [0]:
df_silver_layer=spark.sql(f"select * from {catalog}.{silver_schema}.{data_source}")
display(df_silver_layer)

In [0]:
df_gold=df_silver_layer.select('product_code','month','gross_price')

In [0]:
df_gold.write.format('delta').option('overWriteSchema','true').option('delta.enableChangeDataFeed','true').\
    mode('overwrite').saveAsTable(f"{catalog}.{gold_schema}.sb_dim_{data_source}")

In [0]:
df_gold_price=spark.table(f'{catalog}.{gold_schema}.sb_dim_{data_source}')
display(df_gold_price.limit(10))

In [0]:
from pyspark.sql.window import Window
df_gold_price=df_gold_price.withColumn('year',year('month')
                         ).\
                withColumn('is_zero',when(col('gross_price')==0,1).otherwise(0))

w=Window.partitionBy('product_code','year').orderBy(col('is_zero'),col('month').desc())

df_gold_latest_price=df_gold_price.withColumn('rnk',
                                                     row_number().over(w)
                                                     ).filter(col('rnk')==1)

display(df_gold_latest_price)

In [0]:
df_gold_latest_price=df_gold_latest_price.select('product_code','gross_price','year').withColumnRenamed('gross_price','price_inr')
display(df_gold_latest_price)

In [0]:
from delta.tables import DeltaTable

delta_table=DeltaTable.forName(spark,f'{catalog}.{gold_schema}.dim_gross_price')

delta_table.alias('target').merge(
    source=df_gold_latest_price.alias('source'),
    condition='target.product_code=source.product_code'
).whenMatchedUpdate(set={
    'price_inr':'source.price_inr',
    'year':'source.year'
}).whenNotMatchedInsert(values={
    'product_code':'source.product_code',
    'price_inr':'source.price_inr',
    'year':'source.year'
}).execute()